In [ ]:
"""
Kepler orbit: RK4 energy-drift diagnostic (corrected).

Bug fixes vs. original:
  FIX 1  energy() had +GM/r; the Kepler potential is attractive -> -GM/r.
  FIX 2  the loop stored raw E, whose O(1) orbital swing hides a 1e-7 drift;
         store the relative drift (E - E0)/|E0| instead.
  FIX 3  plt.plot(drift) used the step index; plot against physical time t = i*h.

Validated: E0 = -0.68 exactly for (1, 0, 0, 0.8), NOT -0.87.
"""

import numpy as np
import matplotlib.pyplot as plt

GM = 1.0


# ----------------------------------------------------------------- dynamics
def deriv(state):
    """d/dt [x, y, vx, vy] for the 2-body problem (physics was already correct)."""
    x, y, vx, vy = state
    r = np.sqrt(x**2 + y**2)
    return np.array([vx, vy, -GM * x / r**3, -GM * y / r**3])


def rk4_step(s, h):
    k1 = deriv(s)
    k2 = deriv(s + 0.5 * h * k1)
    k3 = deriv(s + 0.5 * h * k2)
    k4 = deriv(s + h * k3)
    return s + (h / 6.0) * (k1 + 2 * k2 + 2 * k3 + k4)


def energy(s):
    x, y, vx, vy = s
    r = np.sqrt(x**2 + y**2)
    return 0.5 * (vx**2 + vy**2) - GM / r          # FIX 1: potential is negative


def ang_mom(s):
    x, y, vx, vy = s
    return x * vy - y * vx


# ----------------------------------------------------------------- integrate
s0 = np.array([1.0, 0.0, 0.0, 0.8])
h, N = 0.01, 20000

E0, L0 = energy(s0), ang_mom(s0)
a   = -GM / (2 * E0)                                # semi-major axis
ecc = np.sqrt(1 - L0**2 / (GM * a))
P   = 2 * np.pi * a**1.5                            # orbital period

assert abs(E0 - (-0.68)) < 1e-12, f"expected E0 = -0.68, got {E0}"
print(f"E0 = {E0:.6f}   a = {a:.6f}   e = {ecc:.6f}   P = {P:.6f}")
print(f"run length = {N*h:g} time units = {N*h/P:.2f} orbits")

s   = s0.copy()
t   = np.arange(1, N + 1) * h
rel = np.empty(N)
dL  = np.empty(N)
for i in range(N):
    s = rk4_step(s, h)
    rel[i] = (energy(s) - E0) / abs(E0)             # FIX 2: relative drift
    dL[i]  = (ang_mom(s) - L0) / abs(L0)


# --------------------------------------------------- secular vs. oscillatory
slope, intercept = np.polyfit(t, rel, 1)
w  = int(round(P / h))                              # one-period boxcar
sm = np.convolve(rel, np.ones(w) / w, mode="valid") # strips the orbital wiggle

print(f"final relative drift      = {rel[-1]:.3e}")
print(f"linear slope              = {slope:.3e} per unit time")
print(f"orbit-averaged  first/last= {sm[0]:.3e} / {sm[-1]:.3e}")
print(f"angular-momentum drift    = {dL[-1]:.3e}")
verdict = "SECULAR (trending)" if abs(sm[-1] - sm[0]) > 3 * np.std(sm[:w]) \
          else "bounded oscillation"
print("verdict:", verdict)


# --------------------------------------------------------------------- plots
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

ax[0].plot(t, rel, lw=0.7, label="instantaneous")   # FIX 3: time on the x-axis
ax[0].plot(t[:len(sm)] + P / 2, sm, "r", lw=1.6, label="orbit-averaged")
ax[0].axhline(0, color="k", lw=0.5)
ax[0].set_xlabel("time"); ax[0].set_ylabel(r"$(E-E_0)/|E_0|$")
ax[0].set_title(f"RK4 relative energy drift  ({verdict})")
ax[0].legend(fontsize=8)

ax[1].plot(t, rel, lw=0.8)
ax[1].set_xlim(0, 4 * P); ax[1].axhline(0, color="k", lw=0.5)
ax[1].set_xlabel("time"); ax[1].set_ylabel(r"$(E-E_0)/|E_0|$")
ax[1].set_title("first 4 orbits (zoom)")

plt.tight_layout()
plt.show()


# -------------------------------------------------- Week 5 prediction / stub
# PREDICTION (committed before running): leapfrog at the same h will show
# BOUNDED OSCILLATION, not secular growth. It is symplectic, so it exactly
# conserves a shadow Hamiltonian H + O(h^2); true E oscillates in an envelope
# of order h^2 (expect ~1e-4..1e-3 relative here, peaking at pericenter) and
# the orbit-averaged drift stays flat. Angular momentum -> machine precision.
def leapfrog_step(s, h):
    x, y, vx, vy = s
    r3 = (x*x + y*y)**1.5
    vx += 0.5 * h * (-GM * x / r3)                  # kick
    vy += 0.5 * h * (-GM * y / r3)
    x  += h * vx                                    # drift
    y  += h * vy
    r3 = (x*x + y*y)**1.5
    vx += 0.5 * h * (-GM * x / r3)                  # kick
    vy += 0.5 * h * (-GM * y / r3)
    return np.array([x, y, vx, vy])
